# STL Equivalence Checking Pipeline

## Architecture

raw dataset (151 pairs)
        │
        ▼
scale_bound_norm.py          Layer 1: normalise
        │                    - scale real intervals
        │                    - bound unbounded G/F with [0,100000]
        │                    - flag rise/fall → SPOT path
        │                    - flag parametric/symbolic → skip
        ▼
   layer1.csv
        │
        ├─────────────────────────────────────┐
        │                                     │
        ▼                                     ▼
STL_LTL_translation.py              stlsat_checker.py
Layer 2: STL → LTL                  - attempts all 151
- rise/fall → Y operator            - rejects unsupported fragment
- H/O/S → flag unsupported         - exact arithmetic reasoning
- abs() → abstracted prop           - outputs: EQUIV / NON_EQUIV
- strip bounds                        ERROR / TIMEOUT
        │                                     │
        ▼                                     │
   layer2.csv                                 │
        │                                     │
        ▼                                     │
spot_checker.py                               │
- structural equivalence                      │
- handles all 151                             │
- outputs: EQUIV / NON_EQUIV                  │
  ERROR / TIMEOUT                             │
        │                                     │
        ▼                                     ▼
  spot_results.csv              stlsat_results.csv
        │                                     │
        └──────────────┬──────────────────────┘
                       │
                       ▼
              directionality.py
              - merges both results
              - on SPOT NON_EQUIV pairs:
                checks gold ⊆ pred, pred ⊆ gold
              - on STLSat/SPOT disagreements:
                flags arithmetic difference
              - outputs:
                EQUIVALENT (both agree)
                PRED_STRICTER
                PRED_LOOSER
                NON_EQUIVALENT
                ARITHMETIC_DIFF (tools disagree)
                ERROR / SKIP
                       │
                       ▼
              direction_results.csv

## Routing logic
- `stlsat`: bounded operators, integer intervals → STLSat exact check
- `spot`: unbounded G/F (bounded with horizon for SPOT) → LTL translation → SPOT
- `skip`: contains `rise()` or other unsupported operators
